In [3]:
# !pip install -U langchain langchain-milvus google-generativeai pymilvus

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------- ----- 1.8/2.1 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 7.0 MB/s  0:00:00

   ---------------------------------------- 0/5 [greenlet]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- ----------------------- 2/5 [SQLAlchemy]
   ---------------- -----------

In [4]:
# !pip install -U langchain-community

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 12.1 MB/s  0:00:00

   -------- -------------------------------  3/14 [marshmallow]
   ---------------------- -----------------  8/14 [typing-inspect]
   ---------------------------- ----------- 10/14 [pydantic-settings]
   ---------------------------------- ----- 12/14 [aiohttp]
   ---------------------------------- ----- 12/14 [aiohttp]
   ---------------------------------- ----- 12/14 [aiohttp]
   ------------------------------------- -- 13/14 [langchain-community]
   ------------------------------------- -- 13/14 [langchain-community]
   ------------------------------------- -- 13/14 [langchain-community]
   ------------------------------------- -- 13/14 [langchain-community]
   ------------------------------------- -- 13/14 [langchain-community]
   -----------------------------------

In [1]:
# !pip install -U langchain langchain-google-genai

In [2]:
# install docker
# https://milvus.io/docs/install_standalone-windows.md
# install wsl
# install ubuntu
# install milvusdb

In [2]:
# docker-compose up -d


In [2]:
from pymilvus import connections
connections.connect(alias="default", host="127.0.0.1", port="19530")


In [3]:
from pymilvus import utility
print("Collections:", utility.list_collections())


Collections: ['pdf_docs_test', 'milvus_langchain_test']


In [5]:
# direct_gemini_to_milvus.py
import os
import uuid
import google.generativeai as genai            # google-generativeai package
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility


C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# 0. Configure API key
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

In [8]:
# 1. Example data to embed (list of dicts)
documents = [
    {"id": "doc1", "text": "Ini adalah kalimat pertama dalam Bahasa Indonesia.", "meta": {"page": 1, "source": "report.pdf"}},
    {"id": "doc2", "text": "Contoh baris kedua, berisi ringkasan keuangan Q1.", "meta": {"page": 2, "source": "report.pdf"}},
    # ... more ...
]

In [21]:
# 2. Create embeddings in batch (respect rate limits)
def get_gemini_embeddings(texts, model="models/embedding-001"):
    # genai.embed_content returns embeddings -- SDK shape can vary by version.
    # This example assumes embed_content returns a list/dict with 'embedding' vectors.
    response = genai.embed_content(model=model, content=texts)
#     print(response)
    # response structure may be a list; adapt if your SDK version differs
    embeddings = []
    # If response is a dict with 'data' or 'predictions', inspect structure and extract vectors.
    # Example based on typical SDK:
    for item in response["embedding"]:
        embeddings.append(item)
    return embeddings

In [10]:
# 3. Connect to Milvus
connections.connect(host="127.0.0.1", port="19530")

In [11]:
# 4. Create Collection (only once)
collection_name = "pdf_docs_test"
if utility.has_collection(collection_name):
    collection = Collection(collection_name)
else:
    dim = 1536  # set to your embedding dimension (check model); adjust accordingly
    fields = [
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim),
        FieldSchema(name="meta", dtype=DataType.JSON)
    ]
    schema = CollectionSchema(fields, description="PDF doc embeddings")
    collection = Collection(name=collection_name, schema=schema)
    # create index (optional, example IVF_FLAT)
    from pymilvus import Index
    index_params = {"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 1024}}
    collection.create_index(field_name="embedding", index_params=index_params)


In [22]:

# 5. Prepare batches, create embeddings, insert
batch_size = 32
for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    texts = [d["text"] for d in batch]
    embeddings = get_gemini_embeddings(texts)  # list of vectors
    ids = [d["id"] for d in batch]
    metas = [d["meta"] for d in batch]

    # Insert expects column-aligned lists (matching schema order)
    entities = [ids, embeddings, metas]
    collection.insert(entities)

# 6. Flush & load for search
collection.flush()
collection.load()
print("Inserted and loaded collection.")


2025-09-18 15:39:03,554 [ERROR][handler]: RPC error: [batch_insert], <MilvusException: (code=1100, message=the dim (768) of field data(embedding) is not equal to schema dim (1536): invalid parameter[expected=1536][actual=768])>, <Time:{'RPC start': '2025-09-18 15:39:03.545738', 'RPC error': '2025-09-18 15:39:03.554741'}> (decorators.py:140)


MilvusException: <MilvusException: (code=1100, message=the dim (768) of field data(embedding) is not equal to schema dim (1536): invalid parameter[expected=1536][actual=768])>